# 🧬 6조 난임 임신 성공 여부 예측 — 3모델 앙상블 (교육용 초기 노트북)

> ⚠️ **버전 주의:** 이 파일은 사용자가 제공한 실제 초기 단계 노트북의 공개용 정리본입니다. 최종 발표의 A+B+C 세 갈래 블렌딩 결과(OOF 0.741226 / LB 0.7424306)를 재현하는 완성 코드가 **아닙니다**.
> 원본 코드에는 범주형 처리를 위해 train과 test를 결합해 목록을 만드는 부분이 있습니다. 최종 발표에서 설명한 *train 기준 범주 인코딩 원칙*과 달라 대회 규칙 준수/재현 검증에 사용하려면 이 부분을 수정·검토해야 합니다. **실행 검증은 데이터 미제공으로 수행하지 못했습니다.**
> 분석 실습용이며 실제 의료 판단에 사용하지 마세요. 공동 발표 자료에서의 모델 성능을 이 노트북 실행 결과로 소개하면 안 됩니다.


**Claude의 3모델 아이디어를 반영해 처음부터 다시 작성한 오류 방지 버전**입니다.

✅ LightGBM + XGBoost + CatBoost | ✅ 5-Fold 교차검증 | ✅ 파생변수 생성 | ✅ 결측치 구분  
✅ 결과표·한글 그래프 | ✅ 가중평균 앙상블 | ✅ 제출 CSV 자동 다운로드

### 실행 순서 (코드 수정 불필요)
1. [Google Colab](https://colab.research.google.com/)에서 이 `.ipynb` 파일 열기
2. 상단 **런타임 → 모두 실행** 클릭
3. 파일 업로드 창이 뜨면 **`train.csv`, `test.csv`, `sample_submission.csv`**를 한 번에 선택  
   (`train(3).csv` 같은 파일명도 자동 인식)
4. 셀들이 실행되면 마지막에 **`6조_앙상블_최종제출.csv`**가 자동 다운로드됩니다.

> ⚠️ 전체 데이터에서는 3개 모델 × 5회 학습하므로 Colab CPU 환경에서 다소 오래 걸릴 수 있습니다. 중간에 결과가 바로 나오지 않아도 모델 학습 중일 수 있습니다. **코드 실행 전 CSV 3개만 준비하세요.**

> ⚠️ ROC-AUC는 임신 성공률이나 정확도(Accuracy)가 아닙니다. 이 노트북은 교육·연구 실습용 예측 모델입니다. 실제 임상 판단에 사용하면 안 됩니다.

## 1. 필요한 도구 설치 / 한글 글꼴 설정
- Colab에 없는 패키지만 설치합니다.
- CSV 결과는 **UTF-8 BOM**으로 저장해 엑셀에서 한글이 깨질 가능성을 줄입니다.

In [ ]:
import sys, subprocess, importlib.util
requirements = {'lightgbm':'lightgbm', 'xgboost':'xgboost', 'catboost':'catboost',
                'koreanize_matplotlib':'koreanize-matplotlib', 'sklearn':'scikit-learn',
                'pandas':'pandas', 'matplotlib':'matplotlib'}
missing = [pkg for mod, pkg in requirements.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('필요한 패키지 설치:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('✅ 환경 준비 완료')

In [ ]:
import os, re, gc, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    import koreanize_matplotlib  # Colab: 포함된 한글 폰트 사용
    print('✅ 한글 그래프 설정 완료 (koreanize-matplotlib)')
except Exception:
    # koreanize-matplotlib을 설치할 수 없는 실행 환경에서도 시스템 한글 폰트를 탐색
    from matplotlib import font_manager as fm
    available = fm.findSystemFonts()
    fpath = next((f for f in available if 'Nanum' in Path(f).name), None)
    if fpath is None:
        fpath = next((f for f in available if 'NotoSansCJK' in Path(f).name or 'NotoSerifCJK' in Path(f).name), None)
    if fpath:
        fm.fontManager.addfont(fpath)
        plt.rcParams['font.family'] = fm.FontProperties(fname=fpath).get_name()
        print('✅ 한글 그래프 설정 완료 (시스템 글꼴)')
    else:
        print('⚠️ 한글 폰트를 찾지 못했습니다. 설치 단계의 koreanize-matplotlib을 확인하세요.')
plt.rcParams['axes.unicode_minus'] = False

TARGET = '임신 성공 여부'
SEED = 42
SMOKE_TEST = os.environ.get('ENSEMBLE_SMOKE_TEST') == '1'  # 테스트용, 평소에는 False
N_FOLDS = 2 if SMOKE_TEST else 5
LGB_ITER = 50 if SMOKE_TEST else 1800
XGB_ITER = 50 if SMOKE_TEST else 1800
CAT_ITER = 50 if SMOKE_TEST else 1800
PATIENCE = 10 if SMOKE_TEST else 90
print('교차검증:', N_FOLDS, 'Fold /', '검증모드' if SMOKE_TEST else '전체 데이터')

## 2. 파일 자동 인식 및 업로드
- 파일 이름에 `(1)`, `(3)` 등이 붙어 있어도 괜찮습니다.
- **파일 3개를 한 번만 업로드**하면 됩니다.
- 오류가 나면 어떤 파일이 빠졌는지 알려줍니다.

In [ ]:
def find_data_files(folder):
    """이름이 train.csv / train(3).csv / train (3).csv 등인 파일 찾기"""
    folder = Path(folder)
    result = {}
    for key in ['train', 'test', 'sample_submission']:
        candidates = []
        for f in folder.glob('*.csv'):
            if re.fullmatch(re.escape(key) + r'(?:\s*\(\d+\))?\.csv', f.name, flags=re.I):
                candidates.append(f)
        if candidates:
            # train.csv가 있으면 그 파일 우선, 없으면 가장 최근 파일 선택
            result[key] = sorted(candidates, key=lambda f: (f.stem == key, f.stat().st_mtime), reverse=True)[0]
    return result

try:
    import google.colab
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DATA_DIR = Path('/content') if IN_COLAB else Path(os.environ.get('NOTEBOOK_DATA_DIR', '.'))
paths = find_data_files(DATA_DIR)
if IN_COLAB and len(paths) != 3:
    print('📁 아래 업로드 창에서 train / test / sample_submission CSV 3개를 선택하세요.')
    uploaded = colab_files.upload()
    del uploaded
    paths = find_data_files(DATA_DIR)

missing_files = [k for k in ['train', 'test', 'sample_submission'] if k not in paths]
if missing_files:
    raise FileNotFoundError('필수 파일이 없습니다: ' + ', '.join(missing_files) + '\nColab에서 CSV 파일 3개를 업로드하세요.')

def read_csv_safe(filepath):
    for enc in ('utf-8-sig', 'cp949', 'euc-kr'):
        try:
            return pd.read_csv(filepath, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError('CSV 문자 인코딩을 읽지 못했습니다: ' + str(filepath))

train = read_csv_safe(paths['train'])
test = read_csv_safe(paths['test'])
submission = read_csv_safe(paths['sample_submission'])

if TARGET not in train:
    raise ValueError('train 파일에 목표 변수(임신 성공 여부)가 없습니다. train/test 파일을 확인하세요.')
if train[TARGET].isna().any() or not set(train[TARGET].unique()).issubset({0,1}):
    raise ValueError('학습 정답은 0(실패)/1(성공)만 있어야 합니다.')
if 'ID' not in train or 'ID' not in test or 'ID' not in submission or 'probability' not in submission:
    raise ValueError('train/test에는 ID, sample_submission에는 ID와 probability가 있어야 합니다.')
if (not test['ID'].is_unique or not submission['ID'].is_unique or
    set(test['ID']) != set(submission['ID'])):
    raise ValueError('test와 sample_submission의 ID가 일치하지 않습니다.')
if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError('train/test 간 피처 이름이 다릅니다. 올바른 데이터인지 확인하세요.')
if 'ID' in test and 'ID' in train and not train['ID'].is_unique:
    raise ValueError('train ID가 중복되었습니다.')

if SMOKE_TEST:
    train = train.sample(n=min(900, len(train)), random_state=SEED).reset_index(drop=True)
    test = test.iloc[:min(160,len(test))].reset_index(drop=True)
    # 샘플 테스트에서만 제출 ID 축소
    submission = submission[submission['ID'].isin(test['ID'])].reset_index(drop=True)

print('✅ 파일 로드 완료')
for name,path in paths.items(): print(' ', name, '→', path.name)
print('train:', train.shape, '| test:', test.shape, '| submission:', submission.shape)
print('임신 성공률: {:.2%}'.format(train[TARGET].mean()))

## 3. 데이터 탐색 (EDA) + 결측치 처리 원칙
- **수치형 결측치**는 NaN 상태로 유지: 0이나 평균을 억지로 채우지 않습니다.
- **문자형 결측치**는 `__MISSING__`으로 구분합니다.
- 시술상 적용되지 않는 정보(예: DI의 배아 이식 관련 수치)는 **0과 구별**합니다.
- 아래 5가지 성공률 분석은 **관찰 자료의 단순 비교**이며 인과관계를 보여주는 결과는 아닙니다.

In [ ]:
print('📊 학습 데이터 성공/실패 집계')
print(train[TARGET].value_counts().rename(index={0:'실패',1:'성공'}).to_string())
missing_report = (pd.DataFrame({'변수': train.columns,
                                '결측건수': train.isna().sum().values,
                                '결측률(%)': train.isna().mean().values * 100})
                  .sort_values('결측률(%)', ascending=False).reset_index(drop=True))
print('\n결측치 상위 12개')
print(missing_report.head(12).round(2).to_string(index=False))
missing_report.to_csv('6조_결측치_현황.csv', index=False, encoding='utf-8-sig')

In [ ]:
def success_table(df, groups):
    result = (df.groupby(groups, observed=True, dropna=False)[TARGET]
              .agg(시술기록수='size', 성공건수='sum', 성공률='mean').reset_index())
    result['성공률(%)'] = (result.pop('성공률') * 100).round(2)
    return result

analysis = {}
analysis['연령대별'] = success_table(train, ['시술 당시 나이'])
analysis['IVF_DI별'] = success_table(train, ['시술 유형'])
if '이식된 배아 수' in train:
    analysis['IVF_이식배아수별'] = success_table(train[train['시술 유형'] == 'IVF'], ['이식된 배아 수'])
if '배아 이식 경과일' in train:
    analysis['IVF_배아이식경과일별'] = success_table(train[train['시술 유형'] == 'IVF'], ['배아 이식 경과일'])
analysis['연령대_시술유형별'] = success_table(train, ['시술 당시 나이','시술 유형'])

for name, tab in analysis.items():
    print('\n▶', name)
    print(tab.head(20).to_string(index=False))
    tab.to_csv('6조_실제성공률_' + name + '.csv', index=False, encoding='utf-8-sig')

fig, axs = plt.subplots(1, 2, figsize=(12,4))
a = analysis['연령대별'].dropna(subset=['시술 당시 나이'])
axs[0].bar(a['시술 당시 나이'].astype(str), a['성공률(%)'])
axs[0].set_title('시술 당시 연령대별 실제 성공률')
axs[0].set_ylabel('성공률(%)'); axs[0].tick_params(axis='x',rotation=35)
b = analysis['IVF_DI별'].dropna(subset=['시술 유형'])
axs[1].bar(b['시술 유형'].astype(str), b['성공률(%)'])
axs[1].set_title('IVF와 DI 실제 성공률')
axs[1].set_ylabel('성공률(%)')
plt.tight_layout(); plt.savefig('6조_실제성공률_요약.png', dpi=160, bbox_inches='tight'); plt.show()

## 4. 피처 엔지니어링 (AI에게 추가 힌트 만들기)
기존 환자정보 + 과거 시술 횟수 + 시술 유형의 키워드 + 배아 비율을 합쳐 **새 변수**를 만듭니다.

예) 배아 이식 비율 = 이식된 배아 수 ÷ 생성된 배아 수 (분모가 0 또는 결측이면 NaN)

**주의:** `유효 난자 나이`는 나이에 대한 간접 추정용 변수일 뿐, 실제 생물학적 난자 나이 측정값이 아닙니다.

In [ ]:
COUNT_COLS = ['총 시술 횟수','클리닉 내 총 시술 횟수','IVF 시술 횟수','DI 시술 횟수',
              '총 임신 횟수','IVF 임신 횟수','DI 임신 횟수','총 출산 횟수','IVF 출산 횟수','DI 출산 횟수']
AGE_MAP = {'만18-34세':26.0,'만35-37세':36.0,'만38-39세':38.5,
           '만40-42세':41.0,'만43-44세':43.5,'만45-50세':47.5}
DONOR_AGE_MAP = {'만20세 이하':20.0,'만21-25세':23.0,'만26-30세':28.0,
                 '만31-35세':33.0,'만36-40세':38.0,'만41-45세':43.0}
PROCEDURE_TOKENS = ['ICSI','IVF','IUI','ICI','GIFT','FER','BLASTOCYST','AH']
REASON_TOKENS = ['현재 시술용','배아 저장용','난자 저장용','기증용','연구용']

def build_features(df):
    d = df.drop(columns=[TARGET,'ID'], errors='ignore').copy()
    for col in COUNT_COLS:
        if col in d:
            d[col+'_숫자'] = pd.to_numeric(d[col].astype('string').str.extract(r'(\d+)',expand=False), errors='coerce')
    if '시술 당시 나이' in d: d['환자나이_중간값'] = d['시술 당시 나이'].map(AGE_MAP)
    for col in ['난자 기증자 나이','정자 기증자 나이']:
        if col in d: d[col+'_중간값'] = d[col].map(DONOR_AGE_MAP)
    if all(c in d for c in ['난자 출처','난자 기증자 나이_중간값','환자나이_중간값']):
        d['유효난자나이_추정'] = np.where(d['난자 출처']=='기증 제공', d['난자 기증자 나이_중간값'], d['환자나이_중간값'])
    if '특정 시술 유형' in d:
        t = d['특정 시술 유형'].fillna('').astype(str)
        for token in PROCEDURE_TOKENS:
            d['시술문자_'+token] = t.str.contains(token,regex=False).astype('int8')
        d['시술문자_여러시술'] = t.str.contains(':|/',regex=True).astype('int8')
    if '배아 생성 주요 이유' in d:
        t = d['배아 생성 주요 이유'].fillna('').astype(str)
        for token in REASON_TOKENS:
            d['배아이유_'+token] = t.str.contains(token,regex=False).astype('int8')
    ratios = [
        ('배아생성_혼합난자비','총 생성 배아 수','혼합된 난자 수'),
        ('미세주입_생성배아비','미세주입에서 생성된 배아 수','미세주입된 난자 수'),
        ('배아_이식비','이식된 배아 수','총 생성 배아 수'),
        ('배아_저장비','저장된 배아 수','총 생성 배아 수'),
        ('난자_혼합수집비','혼합된 난자 수','수집된 신선 난자 수'),
        ('미세주입_이식비','미세주입 배아 이식 수','이식된 배아 수'),
        ('과거_임신횟수비','총 임신 횟수_숫자','총 시술 횟수_숫자'),
        ('과거_출산횟수비','총 출산 횟수_숫자','총 시술 횟수_숫자')]
    for new,a,b in ratios:
        if a in d and b in d:
            numerator = pd.to_numeric(d[a],errors='coerce')
            denominator = pd.to_numeric(d[b],errors='coerce')
            d[new] = numerator.div(denominator.where(denominator.gt(0)))
    if '배아 이식 경과일' in d and '난자 혼합 경과일' in d:
        d['이식까지_추가경과일'] = d['배아 이식 경과일'] - d['난자 혼합 경과일']
    if '이식된 배아 수' in d:
        d['배아이식여부'] = np.where(d['이식된 배아 수'].isna(), np.nan,
                                      (d['이식된 배아 수']>0).astype('int8'))
    if '배아 이식 경과일' in d:
        d['배아이식_5일이상'] = np.where(d['배아 이식 경과일'].isna(), np.nan,
                                         (d['배아 이식 경과일']>=5).astype('int8'))
    cause_cols = [c for c in d.columns if c.startswith('불임 원인 -')]
    if cause_cols: d['불임원인_세부개수'] = d[cause_cols].sum(axis=1,min_count=1)
    for c in ['임신 시도 또는 마지막 임신 경과 연수','배아 이식 경과일',
              '이식된 배아 수','PGS 시술 여부','난자 해동 경과일']:
        if c in d: d[c+'_결측여부'] = d[c].isna().astype('int8')
    return d.replace([np.inf,-np.inf],np.nan)

# 학습/예측을 합쳐 '범주형 컬럼의 목록'을 일치시킵니다. 정답(target)은 입력하지 않습니다.
Xall = build_features(pd.concat([train.drop(columns=[TARGET]),test], ignore_index=True))
cat_cols = list(Xall.select_dtypes(include=['object','string','category']).columns)
for c in cat_cols:
    Xall[c] = Xall[c].fillna('__MISSING__').astype('category')
for c in Xall.columns.difference(cat_cols):
    Xall[c] = pd.to_numeric(Xall[c], errors='coerce').astype('float32')
X_lgb = Xall.iloc[:len(train)].copy().reset_index(drop=True)
X_lgb_test = Xall.iloc[len(train):].copy().reset_index(drop=True)

# XGBoost에는 범주형 문자를 안정적인 숫자 코드로 변환해서 입력합니다.
X_xgb = X_lgb.copy()
X_xgb_test = X_lgb_test.copy()
for col in cat_cols:
    X_xgb[col] = X_xgb[col].cat.codes.astype('float32')
    X_xgb_test[col] = X_xgb_test[col].cat.codes.astype('float32')

# CatBoost에는 문자열로 바꾼 범주형 데이터가 필요합니다.
X_cb = X_lgb.copy()
X_cb_test = X_lgb_test.copy()
for col in cat_cols:
    X_cb[col] = X_cb[col].astype(str)
    X_cb_test[col] = X_cb_test[col].astype(str)

y = train[TARGET].astype(int).to_numpy()
print('✅ 입력 변수:', len(X_lgb.columns),'개 / 범주형:',len(cat_cols),'개')
print('LightGBM / XGBoost / CatBoost 데이터 준비 완료')
del Xall; gc.collect()

## 5. 5-Fold 교차검증 + 세 가지 AI 학습

모델 세 종류를 각각 5번 학습하고, 매번 보지 않은 데이터로 시험을 봅니다.

- **LightGBM**: 결측값과 범주형 변수를 처리하기 좋은 모델
- **XGBoost**: 범주형 변수를 숫자로 인코딩한 모델
- **CatBoost**: 범주형 변수 처리에 강점이 있는 모델

> 학습에 시간이 걸리는 정상적인 단계입니다. 매 Fold 결과가 출력됩니다.

In [ ]:
splits = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_lgb,y))
models = ['LightGBM', 'XGBoost', 'CatBoost']
oof = {m:np.zeros(len(train), dtype='float64') for m in models}
test_preds = {m:np.zeros(len(test), dtype='float64') for m in models}
score_rows = []
importance = []
START = time.time()

def get_model(name, seed):
    if name == 'LightGBM':
        return lgb.LGBMClassifier(n_estimators=LGB_ITER, learning_rate=0.025,
                 num_leaves=31, max_depth=-1, min_child_samples=90,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 reg_lambda=5, random_state=seed, n_jobs=4, verbosity=-1)
    if name == 'XGBoost':
        return xgb.XGBClassifier(n_estimators=XGB_ITER, learning_rate=0.03,
                 max_depth=6, min_child_weight=15, subsample=0.8,
                 colsample_bytree=0.8, reg_lambda=5, tree_method='hist',
                 eval_metric='auc', early_stopping_rounds=PATIENCE, n_jobs=4,
                 random_state=seed)
    return CatBoostClassifier(iterations=CAT_ITER, learning_rate=0.04,
               depth=6, l2_leaf_reg=5, loss_function='Logloss', eval_metric='AUC',
               random_seed=seed, verbose=False, thread_count=4)

datasets = {'LightGBM':(X_lgb,X_lgb_test),
            'XGBoost':(X_xgb,X_xgb_test),
            'CatBoost':(X_cb,X_cb_test)}
for name in models:
    Xi, Ti = datasets[name]
    print('\n' + '='*30, name, '학습 시작', '='*30,flush=True)
    for k,(tr,va) in enumerate(splits, start=1):
        model = get_model(name, SEED+k)
        kwargs = {}
        if name == 'LightGBM':
            kwargs = {'eval_set':[(Xi.iloc[va],y[va])], 'eval_metric':'auc',
                      'callbacks':[lgb.early_stopping(PATIENCE,verbose=False)]}
        elif name == 'XGBoost':
            kwargs = {'eval_set':[(Xi.iloc[va],y[va])], 'verbose':False}
        else:
            kwargs = {'cat_features':cat_cols, 'eval_set':(Xi.iloc[va],y[va]),
                      'early_stopping_rounds':PATIENCE, 'use_best_model':True}
        model.fit(Xi.iloc[tr], y[tr], **kwargs)
        pv = model.predict_proba(Xi.iloc[va])[:,1]
        pt = model.predict_proba(Ti)[:,1]
        if not np.isfinite(pv).all() or not np.isfinite(pt).all():
            raise ValueError(f'{name}: 예측 확률에 NaN/Inf가 발생했습니다.')
        oof[name][va] = pv
        test_preds[name] += pt / N_FOLDS
        score = roc_auc_score(y[va],pv)
        score_rows.append({'모델':name,'Fold':k,'ROC-AUC':round(score,6)})
        if name == 'LightGBM':
            importance.append(model.booster_.feature_importance(importance_type='gain'))
        print(f'  {k}/{N_FOLDS} Fold: ROC-AUC={score:.5f} / 누적 {((time.time()-START)/60):.1f}분',flush=True)
        del model, pv,pt
        gc.collect()
    print(f'✅ {name} 전체 OOF ROC-AUC = {roc_auc_score(y,oof[name]):.6f}',flush=True)
print('\n세 모델 학습 완료')

## 6. 앙상블 (세 AI의 의견 합치기)

**기본 조합**: LightGBM 40% + XGBoost 30% + CatBoost 30%  
**추가 비교**: OOF에서 10% 단위 가중치를 탐색하고, 개선이 충분한 경우에만 최종 가중치를 변경합니다.

> ⚠️ OOF에서 가중치를 골랐다면 그 OOF 점수는 다소 낙관적일 수 있습니다. 공개 리더보드/새 환자 데이터의 점수를 보장하지 않습니다.

In [ ]:
fixed_weights = np.array([0.4,0.3,0.3])
ensemble_oof_fixed = sum(w*oof[m] for w,m in zip(fixed_weights,models))
fixed_auc = roc_auc_score(y,ensemble_oof_fixed)

best_auc, best_weights = fixed_auc, fixed_weights.copy()
for a in range(11):
    for b in range(11-a):
        weights = np.array([a,b,10-a-b],dtype=float)/10.0
        pred = sum(w*oof[m] for w,m in zip(weights,models))
        auc = roc_auc_score(y,pred)
        if auc > best_auc:
            best_auc, best_weights = auc,weights.copy()

# 같은 OOF에 과도하게 맞추는 위험을 줄이기 위해 미세한 개선이면 고정값 유지
use_searched = best_auc >= fixed_auc + 0.00015
selected_weights = best_weights if use_searched else fixed_weights
selected_oof = sum(w*oof[m] for w,m in zip(selected_weights,models))
selected_auc = roc_auc_score(y,selected_oof)
summary = [{'모델':m,'OOF ROC-AUC':roc_auc_score(y,oof[m])} for m in models]
summary.append({'모델':'고정 앙상블 40:30:30','OOF ROC-AUC':fixed_auc})
summary.append({'모델':'가중치 탐색 최상위 (탐색편향 가능)','OOF ROC-AUC':best_auc})
summary.append({'모델':'최종 앙상블','OOF ROC-AUC':selected_auc})
model_report = pd.DataFrame(summary)
print('\n' + model_report.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
print('\n선택된 가중치:',dict(zip(models,selected_weights)))
print('가중치 탐색 적용:', '예' if use_searched else '아니오 (과대적합 위험으로 고정값 유지)')
model_report.to_csv('6조_모델별_교차검증성능.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(score_rows).to_csv('6조_각Fold_성능.csv',index=False,encoding='utf-8-sig')

if importance:
    imp = pd.Series(np.mean(importance,axis=0),index=X_lgb.columns).sort_values(ascending=False)
    imp.rename('LightGBM 평균 Gain 중요도').to_csv('6조_변수중요도.csv',encoding='utf-8-sig')
    top = imp.head(15).iloc[::-1]
    plt.figure(figsize=(10,6));plt.barh(top.index, top.values)
    plt.title('임신 성공 예측에 사용된 주요 변수 (LightGBM 중요도)')
    plt.xlabel('평균 Gain 중요도')
    plt.tight_layout();plt.savefig('6조_변수중요도.png',dpi=150,bbox_inches='tight');plt.show()
    print('주요 변수 10개:', ', '.join(imp.head(10).index))

## 7. 최종 제출 CSV 자동 생성 ✅

- 모델별로 `test.csv` 전체 데이터에 대한 임신 성공 **확률**을 계산합니다.
- `sample_submission.csv`의 ID 순서에 맞춰 정렬합니다.
- 확률 범위(0~1), 결측치, 행 수를 검사하고 저장합니다.
- Colab에서는 CSV 다운로드가 자동 시작됩니다.

In [ ]:
final_test_pred = sum(w*test_preds[m] for w,m in zip(selected_weights,models))
if len(final_test_pred) != len(test) or not np.isfinite(final_test_pred).all():
    raise ValueError('최종 예측 결과에 길이/결측값 문제가 있습니다.')
if not ((final_test_pred >= 0).all() and (final_test_pred <= 1).all()):
    raise ValueError('확률이 0~1 범위를 벗어났습니다.')

pred_by_id = pd.Series(final_test_pred, index=test['ID'].values)
final_sub = submission[['ID','probability']].copy()
final_sub['probability'] = final_sub['ID'].map(pred_by_id)
if final_sub['probability'].isna().any() or len(final_sub) != len(test):
    raise ValueError('제출 파일과 테스트 데이터 ID 정렬 중 문제가 발생했습니다.')
output_filename = '6조_앙상블_최종제출.csv'
final_sub.to_csv(output_filename,index=False,encoding='utf-8-sig')
print('✅ 제출 파일 생성:',output_filename, '/ 총',len(final_sub),'행')
print('최종 앙상블 검증 ROC-AUC:',round(selected_auc,6))
print(final_sub.head().to_string(index=False))

if IN_COLAB:
    try:
        colab_files.download(output_filename)
        print('✅ 다운로드 요청 완료. 브라우저 다운로드를 확인하세요.')
    except Exception as exc:
        print('⚠️ 다운로드가 시작되지 않았다면 Colab 왼쪽 파일 목록에서 직접 받으세요:',exc)
else:
    print('저장 위치:',Path(output_filename).resolve())

---
### 제출/발표 때 꼭 알아둘 점
1. **LightGBM·XGBoost·CatBoost는 서로 다른 AI 모델**입니다.
2. **앙상블**은 세 모델이 예측한 확률을 가중평균한 것입니다.
3. **5-Fold 교차검증**은 같은 데이터를 5부분으로 나누어 돌아가면서 시험 보는 것입니다.
4. **결측치 처리**는 모르는 숫자를 0으로 임의 변경하지 않고, 문자형 결측은 별도 범주로 구분했습니다.
5. **ROC-AUC 0.74**는 성공 환자의 예측값이 실패 환자보다 높게 나올 확률을 나타내는 지표로, '환자 임신 성공률 74%'가 아닙니다.
6. **한계:** 환자·시술 반복 기록 여부, 의료기관 차이, 확률 보정 여부 등을 검증하지 않았으므로 임상 활용은 불가합니다.

**참고:** 실제 성능 수치는 이 노트북을 실행한 후 출력된 값으로 발표하세요. 기존 Claude 노트북의 값이 그대로 나온다고 보장하지 않습니다.